# 🚀 Nebula GAGAvatar 3D Neural GPU Server (Google Colab)
### خادم المعالجة العصبية ثلاثي الأبعاد لنظام نيبولا (3D Gaussian Splatting + FLAME)

**خطوات التشغيل:**
1. تأكد من تفعيل الـ GPU من: `Runtime -> Change runtime type -> T4 GPU`.
2. شغل الخلايا بالترتيب (Run All أو اضغط Shift + Enter على كل خلية).
3. في نهاية الخلية الأخيرة، سيظهر لك رابط عمومي (مثل `https://xxxx.trycloudflare.com`).
4. انسخ الرابط والصقه في **استوديو نيبولا (Avatar Studio)** في خانة **GAGAvatar Cloud URL** واضغط **حفظ وربط السيرفر**.


In [ ]:
# 1. التأكد من كارت الشاشة GPU
!nvidia-smi


In [ ]:
# 2. تثبيت الحزم والمتطلبات الأساسية
!apt-get update -qq && apt-get install -y -qq ffmpeg ninja-build libgl1
!pip install -q fastapi "uvicorn[standard]" python-multipart httpx scikit-image "imageio[ffmpeg]" huggingface_hub
print("✅ تم تثبيت المتطلبات الأساسية")


In [ ]:
# 3. تحميل مستودع GAGAvatar وتجهيز الموديلات
import os
from pathlib import Path

!git clone --depth 1 https://github.com/xg-chu/GAGAvatar.git vendor/GAGAvatar
!pip install --no-build-isolation -q git+https://github.com/xg-chu/diff-gaussian-rasterization.git || echo "Notice on rasterizer"
!pip install --no-deps -e vendor/GAGAvatar || echo "Local GAGAvatar ready"

os.makedirs("assets/GAGAvatar", exist_ok=True)
print("✅ تم تجهيز مكتبات GAGAvatar")


In [ ]:
# 4. تشغيل خادم GAGAvatar السحابي ونفق Cloudflare المجاني المباشر
import subprocess, time, urllib.request, re

# تحميل أداة Cloudflared لنفق مجاني فوري بدون تسجيل
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# بدء السيرفر في الخلفية
server_proc = subprocess.Popen(["uvicorn", "services.gagavatar_server:app", "--host", "0.0.0.0", "--port", "8090"])
time.sleep(3)

# تشغيل نفق Cloudflare
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8090"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break
    time.sleep(0.5)

if url:
    print("\n" + "="*60)
    print("🎉 تم تشغيل السيرفر العصبي بنجاح!")
    print("🔗 رابط السيرفر العصبي: " + url)
    print("انسخ هذا الرابط وضعه في استوديو الأفاتار في نيبولا")
    print("="*60)
else:
    print("⚠️ برجاء مراجعة مخرجات النفق")
